# Whitrum AI - Training Notebook
**Founder: Oguzhan (Dr0xy-Drawn)**

~350M parameter model based on Qwen architecture.

## Steps:
1. Install dependencies
2. Clone repo & setup
3. Prepare dataset
4. Train with LoRA
5. Save model

In [ ]:
!pip install -q torch transformers accelerate peft datasets tokenizers safetensors
!pip install -q bitsandbytes

In [ ]:
!git clone https://github.com/WhitrumAI/Whitrum-AI.git /content/Whitrum-AI
!cd /content/Whitrum-AI && pip install -e .

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from whitrum import WhitrumConfig, WhitrumForCausalLM

# Create model from config
config = WhitrumConfig(
    vocab_size=60006,
    hidden_size=512,
    intermediate_size=2048,
    num_hidden_layers=12,
    num_attention_heads=8,
    num_key_value_heads=2,
    max_position_embeddings=4096,
)

model = WhitrumForCausalLM(config)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## Load Dataset

In [ ]:
from datasets import load_dataset

# Use a small dataset for demo
# Replace with your own dataset
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:1000]")
print(f"Dataset size: {len(dataset)}")
print(dataset[0])

## Setup LoRA Fine-tuning

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./whitrum-output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=100,
    logging_steps=10,
    save_steps=500,
    fp16=True,
    optim="adamw_torch",
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=None,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

trainer.train()

## Save Model

In [ ]:
model.save_pretrained("./whitrum-350m")
print("Model saved!")

## Test Inference

In [ ]:
prompt = "Hello, I am Whitrum, a language model created by Oguzhan (Dr0xy-Drawn)."
inputs = model.generate(
    model.dumb_encode(prompt) if hasattr(model, 'dumb_encode') else None,
    max_new_tokens=100,
    temperature=0.7,
)
print(inputs)